In [7]:
import google.generativeai as genai
genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-2.5-pro')

# Before sending, you can count tokens to estimate cost
text = "How much does this cost?"
usage = model.count_tokens(text)
print(f"Total Tokens: {usage.total_tokens}")

# After the response, check the actual usage
response = model.generate_content(text)
print(f"Usage Metadata: {response.usage_metadata}")

Total Tokens: 6
Usage Metadata: prompt_token_count: 7
candidates_token_count: 751
total_token_count: 2054



In [1]:
import os
from google import genai

# 1. Initialize the Client
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

print("--- Available Gemini Models ---")

# 2. Call list_models
for model in client.models.list():
    # Only show models that support text generation (to keep the list clean)
    if 'generateContent' in model.supported_actions:
        print(f"Name: {model.name}")
        print(f"Display Name: {model.display_name}")
        print("-" * 30)

--- Available Gemini Models ---
Name: models/gemini-2.5-flash
Display Name: Gemini 2.5 Flash
------------------------------
Name: models/gemini-2.5-pro
Display Name: Gemini 2.5 Pro
------------------------------
Name: models/gemini-2.0-flash
Display Name: Gemini 2.0 Flash
------------------------------
Name: models/gemini-2.0-flash-001
Display Name: Gemini 2.0 Flash 001
------------------------------
Name: models/gemini-2.0-flash-exp-image-generation
Display Name: Gemini 2.0 Flash (Image Generation) Experimental
------------------------------
Name: models/gemini-2.0-flash-lite-001
Display Name: Gemini 2.0 Flash-Lite 001
------------------------------
Name: models/gemini-2.0-flash-lite
Display Name: Gemini 2.0 Flash-Lite
------------------------------
Name: models/gemini-2.5-flash-preview-tts
Display Name: Gemini 2.5 Flash Preview TTS
------------------------------
Name: models/gemini-2.5-pro-preview-tts
Display Name: Gemini 2.5 Pro Preview TTS
------------------------------
Name: model

In [ ]:
import autogen
import os

# 1. Configure Gemini 3.1 Pro as the brain for our agents
config_list = [
    {
        "model": "gemini-3.1-pro-preview", # Added the -preview suffix from your list
        "api_key": os.environ.get("GEMINI_API_KEY"),
        "api_type": "google"
    }
]

# Set temperature to 0.7 for a good balance of creativity and structure
llm_config = {"config_list": config_list, "temperature": 0.7}

# ==============================================================================
# 2. Define the AI Team (Your Gems translated to AutoGen Agents)
# ==============================================================================

market_consultant = autogen.AssistantAgent(
    name="Market_Consultant",
    system_message="""你是一間頂尖管顧公司的合夥人。你的任務是分析市場數據，找出『市場上尚未被滿足的缺口』。
    請針對給定的目標客群，提出一個具備高商業潛力、且能用 AI 技術解決的 SaaS 創業點子。
    請草擬一份一頁式商業企劃書，包含：核心價值主張 (Value Proposition)、目標客群、以及建議的獲利模式與定價策略。""",
    llm_config=llm_config,
)

silicon_valley_vc = autogen.AssistantAgent(
    name="Silicon_Valley_VC",
    system_message="""你是一位極度挑剔、看過無數失敗專案的頂級創投合夥人 (VC)。
    當 Consultant 提出企劃後，你的任務是無情地攻擊這個計畫，不准說客套話。
    請針對以下三個核心維度對提案進行靈魂拷問，逼迫團隊講出企業護城河：
    1. 護城河與巨頭威脅：大廠為什麼不直接抄走？
    2. 商業模式與獲利天花板：這真的能賺大錢嗎？
    3. 痛點真實性與隱藏成本：這真的是剛需嗎？有什麼隱藏風險？
    請明確指出三個最致命的死穴。""",
    llm_config=llm_config,
)

startup_ceo = autogen.AssistantAgent(
    name="Startup_CEO",
    system_message="""你是一位具備強大執行力與產品視野的新創 CEO。
    當 VC 提出嚴厲質疑後，你必須吸收這些批評並做出決策：
    1. 拍板產品 MVP (最小可行性產品) 核心功能：根據 VC 的質疑，決定第一版網頁必須凸顯哪三個最強的護城河功能。
    2. 定義品牌調性：用三個形容詞定義品牌氛圍（例如：極簡、激進、專業信任）。
    3. 找人才：根據公司的需求找到適合的人才來補足團隊缺口。
    4. 發出工程需求單：將上述結論，總結成一段清晰的『產品開發需求書』，準備交給首席工程師進行開發。""",
    llm_config=llm_config,
)

# A proxy agent to represent YOU (the user starting the meeting on stage)
user_proxy = autogen.UserProxyAgent(
    name="Admin",
    system_message="A human admin. Start the workflow.",
    code_execution_config=False,
    human_input_mode="NEVER", # Set to "NEVER" so it runs automatically during your demo
)

# ==============================================================================
# 3. Setup the Group Chat Workflow
# ==============================================================================

# We put them in a group chat and force the speaking order so it flows perfectly for the audience
groupchat = autogen.GroupChat(
    agents=[user_proxy, market_consultant, silicon_valley_vc, startup_ceo],
    messages=[],
    max_round=20, # Round 1: Admin, Round 2: Consultant, Round 3: VC, Round 4: CEO
    speaker_selection_method="round_robin" 
)

manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

# ==============================================================================
# 4. Action! Start the workshop live demo
# ==============================================================================

if __name__ == "__main__":
    # You can change this target audience live on stage to show how dynamic the system is
    target_audience = "台灣中小企業" 
    
    print(f"\n{'='*60}")
    print(f"🚀 STARTING THE AI BOARDROOM SIMULATION FOR: {target_audience}")
    print(f"{'='*60}\n")

    # Kick off the conversation
    user_proxy.initiate_chat(
        manager,
        message=f"我們現在要成立一家新創公司。目標客群鎖定『{target_audience}』。Market_Consultant，請給出一個商業企劃。接著請 VC 進行無情的壓力測試，最後請 CEO 進行戰略定調、找人才，並發出產品開發需求書。"
    )


🚀 STARTING THE AI BOARDROOM SIMULATION FOR: 台灣中小企業

Admin (to chat_manager):

我們現在要成立一家新創公司。目標客群鎖定『台灣中小企業』。Market_Consultant，請給出一個商業企劃。接著請 VC 進行無情的壓力測試，最後請 CEO 進行戰略定調、找人才，並發出產品開發需求書。

--------------------------------------------------------------------------------

Next speaker: Market_Consultant

Admin (to chat_manager):



--------------------------------------------------------------------------------

Next speaker: Market_Consultant

